# Neutral density and how to calculate it
*Contributors*: [Iris Liang](https://github.com/xliang576)

## What is neutral density surface and what is good about it
Simply speaking, neutral density is better approcimation of the surfaces which properties are moved along by mesoscale eddies and turbulence in the ocean than potential density, because it doesn't need a reference pressure level so can align consistently with different depth.  It is "a smooth surface that tangential to potential density layers at each point of reference pressure. An envelop curve of many continually changing reference pressures."

For more detail on neutral surface, ref: [McDougall, 1987](https://journals-ametsoc-org.ezproxy.st-andrews.ac.uk/view/journals/phoc/17/11/1520-0485_1987_017_1950_ns_2_0_co_2.xml) and [Jackett and McDougall, 1997](https://journals-ametsoc-org.ezproxy.st-andrews.ac.uk/view/journals/phoc/27/2/1520-0485_1997_027_0237_andvft_2.0.co_2.xml).


## Python packages for calculation
- [pygamma](https://currents.soest.hawaii.edu/hgstage/pygamma/file/tip/README.md) A python package to calculate neutral surface
    - a guide on [Installing](https://forum.access-hive.org.au/t/neutral-surfaces-calculation-in-python/448/6) pygamma
    - A [recipe](https://github.com/COSIMA/cosima-recipes/blob/main/03-Mains/Neutral_density.ipynb) for pygamma neutral density calculation
    - A debugging tip: when installing, I have problem with "preparing metadata". This could be version generating file has problem, comment out the `tool.setuptools_scm` part and manually change the version file; or could be dependent packages not installed, like fortran-compiler
    - An issue with the package: I got NaN or '0' for results from a few very cold and fresh T-S profiles, for reasons I am not clear. So be careful to check!
- [neutralocean](https://github.com/geoffstanley/neutralocean) this is a package to calculate $w$ surface, which is arguably even better than neutral surface, ref: [paper](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020MS002436)
- [an example](https://neutralocean.readthedocs.io/en/latest/examples.html#ex-eccov4) to use xgcm grid in neutralocean
- [Discussion](https://forum.access-hive.org.au/t/new-python-package-for-neutral-surfaces/250) for two packages 


## Examples using pygamma function gamma_n to calculate

input: insitu temperature, absolute salinity, pressure, longitude, latitude

output: neutral density

But it only takes 2d array, so need a loop get a 3d field 

In [ ]:
import gsw
from pygamma_n import gamma_n
import xarray as xr
import numpy as np

In [ ]:
# loading S and T
def load_S_T(PATH,year):
    ''' loading absolute salinity and in-situ temperature needed for neutral density calculation.'''
    import gsw
    ## load absolute salinity
    S = xr.open_mfdataset(PATH +f'Salt_bsoseI156_{year}_5day.nc',chunks={'time':1})
    S = S.where(S['SALT'] != 0)  # land mask

    ## calculate pressure
    p = gsw.p_from_z(S.Z,S.YC)

    ## load potential temperature
    T = xr.open_mfdataset(PATH +f'{year}/Theta_bsoseI156_{year}_5day.nc',chunks={'time':1})
    ## calculate conservative temperature (for insitu temperature)
    CT =gsw.CT_from_pt(S.SALT,T.THETA)
    ## calculate in-situ temperature
    IT = gsw.t_from_CT(S.SALT,CT,p ).rename('THETA')

    return S, IT 


path = "/gws/ssde/j25a/co2clim/datasets/bSOSE/ITER156/"
year = '2014'
S,T = load_S_T(path,year)



In [ ]:
## For a 2d (x,z) section
latitude = -60
S_i = S.sel(YC = latitude,method='nearest')
T_i = T.sel(YC = latitude,method='nearest')
P = xr.DataArray(np.full(S_i['SALT'].shape, np.nan), dims=S_i['SALT'].dims, coords=S_i['SALT'].coords )


gamma , dg_lo, dg_hi   = pygamma_n.gamma_n(S_i['SALT'].transpose(),T_i['THETA'].transpose(),\
                                          P.transpose(),lon = S_i['XC'], lat = latitude)  
gamma = xr.DataArray(gamma.T, dims = S_i['SALT'].dims , coords = S_i['SALT'].coords, name = 'gamma')
gamma.attrs['long_name'] = 'neutral density'
gamma.attrs['units'] = 'kg/m3'

In [ ]:
## For a 3d (x,y,z) section, this is a function that I wrote and used

def gamma_xyz(S_t,T_t):
    ''' Calculate 3D(x,y,z) neutral density from salinity and temperature'''

    S_t = S_t.SALT
    
    def gamma_n_safe(S, T, P, lon=None, lat=None):
        try:
            g, _, _ = gamma_n(S, T, P, lon=lon, lat=lat)
            return g
        except RuntimeError as e:
            if "negative scv" in str(e).lower():
                return np.full_like(S, np.nan)
            raise

    for y in S_t['YC']:
        S_t_y = S_t.sel(YC = y)
        T_t_y = T_t.sel(YC = y)
        P = gsw.p_from_z(z = S_t_y['Z'], lat = S_t_y['YC'])
        
        gamma_y = xr.apply_ufunc(
            gamma_n_safe,
            S_t_y.transpose(), T_t_y.transpose(), P,
            kwargs={'lon': S_t_y.XC, 'lat': S_t_y.YC},
            input_core_dims=[[ "XC","Z"], ["XC","Z"], ["Z"]],
            output_core_dims=[[ "XC","Z"]],
            vectorize=False,      # gamma_n already handles arrays, no need to vectorize
            dask="parallelized",  # enable dask parallel execution if data are chunked
            output_dtypes=[float],
        ).compute()

        if y == S_t['YC'][0]:
            gamma_t = gamma_y.expand_dims('YC')
        else:
            gamma_t = xr.concat([gamma_t, gamma_y.expand_dims('YC')], dim='YC')

    gamma_t.attrs['long_name'] = 'neutral density'
    gamma_t.attrs['units'] = 'kg m-3'
    gamma_t = gamma_t.rename('gamma')
    gamma_t = gamma_t.where(gamma_t != 0)  # this is because I had a few 0 results from very cold and fresh water, and I simply mask it out
    gamma_t.close()
    return gamma_t